# Test run: cyclic peptide binder vs PDL1 (no AlphaFold 3)

This notebook mirrors the **PDL1** example from the project README (programmed death-ligand 1 binder design). The README’s primary worked example is a protein–protein binder against the extracellular domain sequence shown in the Boltz section.

## AlphaFold 3 and commercial use

Google DeepMind’s **AlphaFold 3** terms restrict certain uses (including many commercial applications). This notebook **does not** invoke AF3: we omit `--use_alphafold3_validation` and `--use_msa_for_af3`. Design and in-pipeline scoring still use **Boltz** (diffusion + structure prediction) and the rest of the Protein Hunter stack as implemented in `boltz_ph/`.

After design, this notebook can run **Chai-1 cross-validation** on high-ipTM YAMLs (`--downstream_validation chai`), i.e. re-predict complexes with Chai (not Boltz) while keeping the same **PyRosetta** post-processing as AF3 validation. Set `DOWNSTREAM_VALIDATION = "none"` in the command cell to skip CV for a faster smoke test. Chai weights should be available (see README / `./setup.sh`).

## Prerequisites

- Run cells with the Jupyter **kernel working directory** inside this repo (or any subfolder); `find_repo_root` walks parents to locate `boltz_ph/design.py`.
- The subprocess uses **`cwd=REPO_ROOT`** (repo root), not `boltz_ph/`. The pipeline calls LigandMPNN with `./LigandMPNN/run.py`, which is relative to the process cwd—matching the README pattern `python boltz_ph/design.py` from the repo root.
- GPU recommended; override with env var `PROTEIN_HUNTER_GPU` or edit `GPU_ID` in the code cell.
- Conda env / `./setup.sh` completed per README.

## 1. Paths and target sequence

Canonical **PDL1** target sequence from the README Boltz examples (single-chain protein target; design chain is **A**, target **B** in pipeline conventions).

In [1]:
import os
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upward from start until boltz_ph/design.py exists."""
    for p in [start, *start.parents]:
        if (p / "boltz_ph" / "design.py").is_file():
            return p
    raise FileNotFoundError(
        "Could not find boltz_ph/design.py. Open the notebook from Protein-Hunter "
        "or set REPO_ROOT manually to the repo root."
    )


# Jupyter: kernel cwd is usually the folder containing the notebook
REPO_ROOT = find_repo_root(Path.cwd())

DESIGN_SCRIPT = REPO_ROOT / "boltz_ph" / "design.py"

# README Boltz example: human PD-L1 extracellular region (one-letter code)
PDL1_TARGET_SEQ = (
    "AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE"
)

GPU_ID = int(os.environ.get("PROTEIN_HUNTER_GPU", "0"))

print("REPO_ROOT:", REPO_ROOT)
print("design.py:", DESIGN_SCRIPT)
print("PDL1 length:", len(PDL1_TARGET_SEQ))

REPO_ROOT: /home/kyle/Protein-Hunter
design.py: /home/kyle/Protein-Hunter/boltz_ph/design.py
PDL1 length: 122


## 2. Build a minimal **cyclic peptide** design command (Boltz)

This follows the README’s **cyclic peptide binder** recipe: short length window, `--cyclic`, high `iptm` threshold. Values are reduced for a quick **smoke test** (`num_designs`, `num_cycles`); increase them for real campaigns.

**Intentionally omitted** (AF3 / AF3-input-related):

- `--use_alphafold3_validation`
- `--use_msa_for_af3`

**Cross-validation (Boltz designs):** the next cell sets `DOWNSTREAM_VALIDATION` to `"chai"` (Chai re-folds high-ipTM YAMLs) or `"none"`. On a **Chai** design notebook you would use `--downstream_validation boltz` instead; that flag is not used here because this path runs `boltz_ph/design.py`.

In [2]:
RUN_NAME = "notebook_PDL1_cyclic_peptide_no_af3_smoke"
OUT_DIR = REPO_ROOT / "outputs" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cross-validate Boltz-designed hits with Chai-1 (true out-of-stack CV). Use "none" to skip.
DOWNSTREAM_VALIDATION = "chai"  # "none" | "chai"

cmd = [
    "python",
    str(DESIGN_SCRIPT),
    "--num_designs", "100",
    "--num_cycles", "3",
    "--protein_seqs", PDL1_TARGET_SEQ,
    "--msa_mode", "mmseqs",
    "--gpu_id", str(GPU_ID),
    "--name", RUN_NAME,
    "--min_protein_length", "10",
    "--max_protein_length", "30",
    "--high_iptm_threshold", "0.8",
    "--percent_X", "100",
    "--save_dir", str(OUT_DIR),
    "--plot",
]
if DOWNSTREAM_VALIDATION and DOWNSTREAM_VALIDATION != "none":
    cmd += ["--downstream_validation", DOWNSTREAM_VALIDATION]

print("Working directory for subprocess (repo root):", REPO_ROOT)
print("DOWNSTREAM_VALIDATION:", DOWNSTREAM_VALIDATION)
print("Command:\n", " \\\n".join(cmd))

Working directory for subprocess (repo root): /home/kyle/Protein-Hunter
DOWNSTREAM_VALIDATION: chai
Command:
 python \
/home/kyle/Protein-Hunter/boltz_ph/design.py \
--num_designs \
100 \
--num_cycles \
3 \
--protein_seqs \
AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE \
--msa_mode \
mmseqs \
--gpu_id \
0 \
--name \
notebook_PDL1_cyclic_peptide_no_af3_smoke \
--min_protein_length \
10 \
--max_protein_length \
30 \
--high_iptm_threshold \
0.8 \
--percent_X \
100 \
--save_dir \
/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke \
--plot \
--downstream_validation \
chai


## 3. Run design

Run with **`cwd=REPO_ROOT`**. Python still puts `boltz_ph/` on `sys.path` as the script directory for `design.py`, while `./LigandMPNN/run.py` resolves correctly under the repo root.

In [3]:
proc = subprocess.run(
    cmd,
    cwd=str(REPO_ROOT),
    env={**os.environ},
    check=False,
)
print("Exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError("design.py failed; scroll up for stderr/stdout from the subprocess.")

Design Configuration:
gpu_id                        : 0
grad_enabled                  : False
name                          : notebook_PDL1_cyclic_peptide_no_af3_smoke
mode                          : binder
num_designs                   : 100
num_cycles                    : 3
cyclic                        : False
min_protein_length            : 10
max_protein_length            : 30
seq                           : 
refiner_mode                  : False
protein_seqs                  : AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE
msa_mode                      : mmseqs
ligand_smiles                 : 
ligand_ccd                    : 
nucleic_type                  : dna
nucleic_seq                   : 
template_path                 : 
template_cif_chain_id         : 
diffuse_steps                 : 200
recycling_steps               : 3
boltz_model_version           : boltz2
boltz_model_path              : ~/.boltz

/home/kyle/miniconda3/envs/proteinhunter/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0


✅ ProteinHunter_Boltz initialized.
Data dictionary:
 {'sequences': [{'protein': {'id': ['A'], 'sequence': 'X', 'msa': 'empty', 'cyclic': False}}, {'protein': {'id': ['B'], 'sequence': 'AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE', 'msa': '/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/0_protein_hunter_design/B_env/msa.npz'}}]}

=== Starting Design Run 0/99 ===
Binder initial sequence length: 22
Empty MSA found; using single sequence mode.

--- Run 0, Cycle 1 ---
Empty MSA found; using single sequence mode.
ipTM: 0.94 pLDDT: 0.86 iPLDDT: 0.79 Alanine count: 0
✅ Saved run 0 cycle 1 YAML.
✅ Copied PDB to /home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/high_iptm_pdb/notebook_PDL1_cyclic_peptide_no_af3_smoke_run_0_cycle_1_structure.pdb
✅ Appended high-ipTM metrics to /home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/summary_high_ip

/home/kyle/Protein-Hunter/boltz_ph/../utils/metrics.py:163: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  ca_coords_tensor = torch.tensor(ca_coords)


iptm: 0.28, plddt: 0.8, rg: 8.9, i_pae: 12.9, apo_holo_rmsd: 4.0
iptm: 0.17, plddt: 0.8, rg: 8.5, i_pae: 14.2, apo_holo_rmsd: 4.4
iptm: 0.34, plddt: 0.9, rg: 9.3, i_pae: 10.4, apo_holo_rmsd: 3.5
iptm: 0.50, plddt: 0.9, rg: 9.0, i_pae: 9.5, apo_holo_rmsd: 4.3
iptm: 0.69, plddt: 0.9, rg: 10.3, i_pae: 7.5, apo_holo_rmsd: 2.0
iptm: 0.47, plddt: 0.9, rg: 7.1, i_pae: 8.6, apo_holo_rmsd: 5.1
iptm: 0.52, plddt: 0.9, rg: 12.0, i_pae: 7.6, apo_holo_rmsd: 2.4
iptm: 0.60, plddt: 0.9, rg: 10.3, i_pae: 6.3, apo_holo_rmsd: 0.6
iptm: 0.69, plddt: 0.9, rg: 7.6, i_pae: 5.2, apo_holo_rmsd: 0.2
iptm: 0.45, plddt: 0.8, rg: 8.3, i_pae: 10.5, apo_holo_rmsd: 5.7
iptm: 0.22, plddt: 0.8, rg: 8.2, i_pae: 13.6, apo_holo_rmsd: 5.9
iptm: 0.21, plddt: 0.8, rg: 10.0, i_pae: 14.0, apo_holo_rmsd: 6.7
iptm: 0.49, plddt: 0.9, rg: 8.7, i_pae: 10.6, apo_holo_rmsd: 5.6
iptm: 0.21, plddt: 0.8, rg: 7.1, i_pae: 13.3, apo_holo_rmsd: 6.3
iptm: 0.26, plddt: 0.8, rg: 8.5, i_pae: 12.9, apo_holo_rmsd: 1.7
iptm: 0.25, plddt: 0.9, rg:

## 4. Inspect outputs

Under `save_dir`: `summary_all_runs.csv`, and when thresholds are met, `high_iptm_yaml` / `high_iptm_pdb` (see README “Successful Designs”).

If `DOWNSTREAM_VALIDATION` was `"chai"`, also check `save_dir/1_af3_rosetta_validation/` for Chai holo/apo CIFs, converted PDBs, Rosetta `rosetta_energy.csv`, and filtered CSVs from the same Rosetta path as AF3 validation (folder name is historical).

In [4]:
import pandas as pd
from IPython.display import display

summary = OUT_DIR / "summary_all_runs.csv"
if summary.is_file():
    display(pd.read_csv(summary).head())
else:
    print("No summary yet:", summary)

for sub in ["high_iptm_yaml", "high_iptm_pdb", "high_iptm_cif", "1_af3_rosetta_validation"]:
    p = OUT_DIR / sub
    if p.is_dir():
        files = list(p.iterdir())
        print(f"{sub}: {len(files)} files")
    else:
        print(f"{sub}: (missing)")

,run_id,cycle_0_iptm,cycle_0_plddt,cycle_0_iplddt,cycle_0_alanine,cycle_0_seq,cycle_1_iptm,cycle_1_plddt,cycle_1_iplddt,cycle_1_alanine,...,cycle_2_seq,cycle_3_iptm,cycle_3_plddt,cycle_3_iplddt,cycle_3_alanine,cycle_3_seq,best_iptm,best_cycle,best_plddt,best_seq
0,0,0.361518,0.850981,0.739904,0,XXXXXXXXXXXXXXXXXXXXXX,0.936620,0.864589,0.791258,0,...,GPSAERLRKELGSVADKILPSD,0.892780,0.924774,0.908162,1,SERLERLREELGEKAEEILPEE,0.936620,1.0,0.864589,SSYSGRVSKNLGGVGSLLDSSD
1,1,0.369961,0.858513,0.763066,0,XXXXXXXXXXXXXXXXXXXXXX,0.943249,0.850827,0.781048,1,...,MEVVDRTQETLEFLQRVLAGQE,0.927709,0.918702,0.887592,5,DPSADRSAEVAELLAKRLAGEL,0.943545,2.0,0.954886,MEVVDRTQETLEFLQRVLAGQE
2,2,0.410780,0.897780,0.809112,0,XXXXXXXXXXXXX,0.925066,0.958065,0.945870,3,...,SRLEAVAAGKVTA,0.834105,0.919321,0.898673,5,SELRAKAEAARLA,NaN,NaN,NaN,NaN
3,3,0.386767,0.895781,0.787912,0,XXXXXXXXXXXXX,0.892895,0.935138,0.906842,3,...,SRLEEKLAEARAA,0.911163,0.965669,0.965100,4,SRLEKKLAEARAA,NaN,NaN,NaN,NaN
4,4,0.408742,0.900746,0.793082,0,XXXXXXXXXXXX,0.858860,0.923343,0.866129,0,...,STLVLEVPPVGS,0.837783,0.916502,0.857923,0,SVQVLELPPRER,0.858860,1.0,0.923343,SDLSLSVSGSGS


high_iptm_yaml: 240 files
high_iptm_pdb: 240 files
high_iptm_cif: (missing)
1_af3_rosetta_validation: 7 files


In [5]:
from IPython.display import Markdown, display
import pandas as pd

display(Markdown("## 5. Cross-validation review (Chai + Rosetta)"))

cv_dir = OUT_DIR / "1_af3_rosetta_validation"
chai_scores_path = cv_dir / "03_af_pdb_success" / "high_iptm_confidence_scores.csv"
boltz_hits_path = OUT_DIR / "summary_high_iptm.csv"
rosetta_csv = cv_dir / "03_af_pdb_success" / "rosetta_energy.csv"
rosetta_out = cv_dir / "af_pdb_rosetta_success"

if str(DOWNSTREAM_VALIDATION).lower() == "none":
    display(Markdown("_Cross-validation was skipped (`DOWNSTREAM_VALIDATION == \"none\"`)._"))
elif not cv_dir.is_dir():
    display(Markdown(f"_No folder `{cv_dir}` yet — run the design cell with Chai CV enabled._"))
else:
    display(Markdown(f"**Validation root:** `{cv_dir}`"))

    # --- Boltz (design stack) vs Chai (CV stack) on the same high-ipTM saves ---
    if chai_scores_path.is_file():
        chai = pd.read_csv(chai_scores_path)
        chai["cv_key"] = chai["file"].astype(str).str.replace(r"_model\.cif$", "", regex=True)
        if boltz_hits_path.is_file():
            boltz = pd.read_csv(boltz_hits_path)
            boltz["cv_key"] = (
                boltz["yaml_filename"].astype(str).str.replace(r"\.yaml$", "", regex=True)
            )
            boltz_cmp = boltz.rename(
                columns={
                    "iptm": "iptm_boltz_design",
                    "plddt": "plddt_boltz_design",
                    "iplddt": "iplddt_boltz_design",
                }
            )
            chai_cmp = chai.rename(
                columns={"iptm": "iptm_chai_cv", "plddt": "mean_plddt_chai_atoms"}
            )
            join_cols = ["cv_key", "iptm_chai_cv", "mean_plddt_chai_atoms"]
            if "rmsd" in chai_cmp.columns:
                join_cols.append("rmsd")
            merged = boltz_cmp.merge(chai_cmp[join_cols], on="cv_key", how="outer")
            show = merged.sort_values("cv_key")[
                [
                    "cv_key",
                    "run_id",
                    "cycle",
                    "iptm_boltz_design",
                    "plddt_boltz_design",
                    "iptm_chai_cv",
                    "mean_plddt_chai_atoms",
                    "sequence",
                ]
                + (["rmsd"] if "rmsd" in merged.columns else [])
            ]
            display(Markdown("**Boltz design metrics vs Chai re-prediction** (same YAML / design id)"))
            display(show)
        else:
            display(Markdown("**Chai confidence (holo)** — no `summary_high_iptm.csv` to join"))
            display(chai.drop(columns=["cv_key"], errors="ignore"))
    else:
        display(Markdown(f"_No Chai scores file yet: `{chai_scores_path}`_"))

    if rosetta_csv.is_file():
        display(Markdown("**Rosetta interface scores** (structures converted from Chai holo CIFs)"))
        display(pd.read_csv(rosetta_csv))

    for label, fname in [
        ("Rosetta pass + extra filters", "success_designs.csv"),
        ("Rosetta / post filters — failed", "failed_designs.csv"),
    ]:
        p = rosetta_out / fname
        if not p.is_file():
            continue
        try:
            df_post = pd.read_csv(p)
        except pd.errors.EmptyDataError:
            display(Markdown(f"**{label}** (`{fname}`) — _empty or whitespace-only CSV_"))
            continue
        display(Markdown(f"**{label}** (`{fname}`)"))
        if df_post.empty:
            display(Markdown("_No data rows._"))
        else:
            display(df_post)

    display(
        Markdown(
            "**Artifacts (paths)**  \n"
            f"- Chai holo CIFs: `{cv_dir / '02_design_final_af3'}`  \n"
            f"- Chai apo CIFs: `{cv_dir / '02_design_final_af3_apo'}`  \n"
            f"- Converted holo PDBs: `{cv_dir / '03_af_pdb_success'}`  \n"
            f"- Zip bundle: `{cv_dir / 'af_pdb_rosetta_success.zip'}`"
        )
    )


## 5. Cross-validation review (Chai + Rosetta)

**Validation root:** `/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/1_af3_rosetta_validation`

**Boltz design metrics vs Chai re-prediction** (same YAML / design id)

,cv_key,run_id,cycle,iptm_boltz_design,plddt_boltz_design,iptm_chai_cv,mean_plddt_chai_atoms,sequence,rmsd
0,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,0,1,0.805424,0.899909,0.257311,0.881850,SESGARFTVDGGRAVVRLEG,NaN
1,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,0,1,0.936620,0.864589,0.257311,0.881850,SSYSGRVSKNLGGVGSLLDSSD,NaN
2,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,0,2,0.906025,0.923874,0.328798,0.858147,GEGGVEIVTVDGKERVKLTD,NaN
3,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,0,2,0.871553,0.954718,0.328798,0.858147,GPSAERLRKELGSVADKILPSD,NaN
4,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,0,3,0.902633,0.921923,0.432656,0.877599,GPGGVEIVTVDGRPQVRLLP,NaN
...,...,...,...,...,...,...,...,...,...
238,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,98,1,0.947906,0.907978,0.404664,0.861310,GDELARRDLEGAKELDMKFGAES,1.549398
239,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,98,3,0.912531,0.921623,0.598538,0.869105,MDPEALKALQEAKERARKELELE,0.600608
240,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,9,1,0.879722,0.893225,0.135327,0.879194,SILGSSSSSSS,3.796247
241,notebook_PDL1_cyclic_peptide_no_af3_smoke_run_...,9,2,0.893334,0.919480,0.177965,0.900269,SVLGTSSPDLP,5.157659


**Rosetta interface scores** (structures converted from Chai holo CIFs)

,PDB,Model,binder_score,surface_hydrophobicity,interface_sc,interface_packstat,interface_dG,interface_dSASA,interface_dG_SASA_ratio,interface_fraction,interface_hydrophobicity,interface_nres,interface_interface_hbonds,interface_hbond_percentage,interface_delta_unsat_hbonds,interface_delta_unsat_hbonds_percentage
0,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-374.29,0.37,0.71,0.72,-32.54,1080.06,-3.01,16.52,60.00,15,1,6.67,2.0,13.33
1,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-375.86,0.36,0.65,0.72,-31.58,1109.10,-2.85,17.13,44.44,18,3,16.67,3.0,16.67
2,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-378.39,0.37,0.66,0.71,-31.90,1067.48,-2.99,16.26,50.00,16,2,12.50,5.0,31.25
3,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-388.47,0.33,0.56,0.59,-31.72,1255.84,-2.53,19.36,38.89,18,4,22.22,1.0,5.56
4,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-384.09,0.34,0.67,0.64,-27.78,1045.34,-2.66,16.36,50.00,14,1,7.14,3.0,21.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-374.29,0.35,0.71,0.62,-21.02,1096.94,-1.92,17.05,43.75,16,0,0.00,3.0,18.75
239,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-383.36,0.35,0.72,0.68,-35.10,1216.99,-2.88,18.46,44.44,18,4,22.22,3.0,16.67
240,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-388.48,0.35,0.73,0.70,-40.27,1184.31,-3.40,17.74,47.06,17,3,17.65,1.0,5.88
241,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-366.25,0.34,0.44,0.69,-14.55,861.57,-1.69,12.75,46.15,13,3,23.08,0.0,0.00


**Rosetta pass + extra filters** (`success_designs.csv`) — _empty or whitespace-only CSV_

**Rosetta / post filters — failed** (`failed_designs.csv`)

,PDB,Model,binder_score,surface_hydrophobicity,interface_sc,interface_packstat,interface_dG,interface_dSASA,interface_dG_SASA_ratio,interface_fraction,...,interface_hbond_percentage,interface_delta_unsat_hbonds,interface_delta_unsat_hbonds_percentage,apo_holo_rmsd,iptm,plddt,i_pae,rg,aa_seq,failure_reason
0,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-388.47,0.33,0.56,0.59,-31.72,1255.84,-2.53,19.36,...,22.22,1.0,5.56,6.460632,0.334056,0.893504,11.101408,9.925646,SEVQKEIEKRLAELDGTFKPRSSS,Does not pass iptm/plddt/rg/i_pae/rmsd thresho...
1,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-381.50,0.33,0.63,0.62,-37.95,1564.79,-2.43,25.14,...,18.18,3.0,13.64,5.758311,0.457126,0.862708,11.349081,11.343291,EDLSLSSSEVSEADLLSKVN,Does not pass iptm/plddt/rg/i_pae/rmsd thresho...
2,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-378.55,0.32,0.73,0.63,-35.01,1131.49,-3.09,17.49,...,26.67,1.0,6.67,7.155145,0.233066,0.845525,12.185825,7.523556,SRFSERLTNGATADFREEGS,Does not pass iptm/plddt/rg/i_pae/rmsd thresho...
3,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-384.38,0.34,0.67,0.57,-40.67,1474.85,-2.76,23.71,...,30.00,3.0,15.00,3.985871,0.283696,0.815984,12.927531,8.874069,SLREEVLARLTLGADNGSSSIRVFSDSSG,Does not pass iptm/plddt/rg/i_pae/rmsd thresho...
4,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-374.71,0.32,0.57,0.60,-26.57,1279.13,-2.08,19.08,...,23.53,3.0,17.65,4.435578,0.165246,0.836400,14.176882,8.513286,PPGKGVPFKGVFDSEGRGTAAAS,Does not pass iptm/plddt/rg/i_pae/rmsd thresho...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-374.29,0.35,0.71,0.62,-21.02,1096.94,-1.92,17.05,...,0.00,3.0,18.75,3.246420,0.271862,0.869352,12.138578,7.947844,SSVVEAILNSPLRGPS,surface_hydrophobicity >= 0.35; interface_inte...
239,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-383.36,0.35,0.72,0.68,-35.10,1216.99,-2.88,18.46,...,22.22,3.0,16.67,0.559243,0.533758,0.895266,6.611420,8.602128,SELDEIIAKLKAASKSLL,surface_hydrophobicity >= 0.35
240,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-388.48,0.35,0.73,0.70,-40.27,1184.31,-3.40,17.74,...,17.65,1.0,5.88,0.413013,0.829163,0.913126,3.847862,8.057977,MSSQEVRDYILKLAEAEK,surface_hydrophobicity >= 0.35; interface_inte...
241,/home/kyle/Protein-Hunter/outputs/notebook_PDL...,relax_notebook_PDL1_cyclic_peptide_no_af3_smok...,-366.25,0.34,0.44,0.69,-14.55,861.57,-1.69,12.75,...,23.08,0.0,0.00,3.583035,0.204159,0.891034,12.451419,7.537175,SGLPFTADRTS,interface_sc <= 0.55; interface_interface_hbon...


**Artifacts (paths)**  
- Chai holo CIFs: `/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/1_af3_rosetta_validation/02_design_final_af3`  
- Chai apo CIFs: `/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/1_af3_rosetta_validation/02_design_final_af3_apo`  
- Converted holo PDBs: `/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/1_af3_rosetta_validation/03_af_pdb_success`  
- Zip bundle: `/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/1_af3_rosetta_validation/af_pdb_rosetta_success.zip`